# OPE Validation on the Open Bandit Dataset

Real-world validation of scikit-rec's off-policy evaluation (OPE) estimators using the [Open Bandit Dataset (OBD)](https://research.zozo.com/data.html) [Saito et al., 2021]. OBD is a logged bandit dataset from ZOZOTOWN with two policies running in parallel over the same impressions — a uniform-random logger and a Bernoulli Thompson Sampling (BTS) logger — so the on-policy CTR of one policy can serve as ground truth for off-policy estimates computed from the other policy's logs.

**Estimators evaluated:** `IPSEvaluator`, `SNIPSEvaluator`, `DREvaluator`, `DirectMethodEvaluator`, `ReplayMatchEvaluator`.

**Reported metrics per estimator:** absolute bias, variance over K bootstrap subsamples, and relative RMSE √(E[(V̂/V − 1)²]) — the standard OBD reporting convention.

**Cross-check:** the same arrays are passed to `obp`'s reference IPS / SNIPS / DR / DM implementations; the two libraries should agree to within ~1%.

**Data:** uses the OBD sample release bundled inside the `obp` package (~10K rows per campaign). The full 26M-row release at https://research.zozo.com/data.html can be substituted by pointing `DATA_DIR` at the extracted archive.

## 1. Imports

In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from skrec.evaluator.direct_method import DirectMethodEvaluator
from skrec.evaluator.doubly_robust import DREvaluator
from skrec.evaluator.inverse_propensity_score import IPSEvaluator
from skrec.evaluator.replay_match import ReplayMatchEvaluator
from skrec.evaluator.snips import SNIPSEvaluator
from skrec.metrics.datatypes import RecommenderMetricType

DATA_DIR = Path("data/obd")
DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(0)
print("Imports OK")

Imports OK


## 2. Cache OBD sample

The `obp` package bundles the OBD small-sample release inside its install tree. Copy it to `data/obd/` on first run so the notebook is self-contained and can later be pointed at the full release with no other code changes.

In [2]:
try:
    import obp
except ImportError as exc:
    raise ImportError(
        "The Open Bandit Pipeline (`obp`) package is required for this notebook. Install it with: pip install obp"
    ) from exc

OBP_BUNDLED_OBD = Path(obp.__path__[0]) / "dataset" / "obd"

if not DATA_DIR.exists():
    if not OBP_BUNDLED_OBD.exists():
        raise FileNotFoundError(
            f"obp does not bundle OBD sample at {OBP_BUNDLED_OBD}. "
            "Either reinstall obp or download the full release from "
            "https://research.zozo.com/data.html and extract to data/obd/."
        )
    print(f"Caching OBD sample: {OBP_BUNDLED_OBD} -> {DATA_DIR}")
    shutil.copytree(OBP_BUNDLED_OBD, DATA_DIR)
else:
    print(f"OBD sample already cached at {DATA_DIR}")

OBD sample already cached at data/obd


## 3. Load logged-policy data

OBD ships three campaigns (`men`, `women`, `all`) × two logging policies (`random`, `bts`). For each campaign we have parallel logs for both policies over the same impressions, which is what makes the on-policy CTR of one policy usable as ground truth for an off-policy estimate computed from the other policy's logs.

In [3]:
CAMPAIGN = "all"  # one of 'men', 'women', 'all'


def load_obd_logs(policy: str, campaign: str, data_dir: Path) -> dict:
    """Load OBD impression logs directly from the bundled CSVs.

    Bypasses ``obp.dataset.OpenBanditDataset`` because obp 0.4.1's preprocess
    step uses the deprecated ``DataFrame.drop("col", 1)`` positional axis arg,
    which raises under pandas 2.x. Reading the CSVs ourselves is also more
    transparent for the paper than wrapping obp's loader as a black box.
    """
    df = pd.read_csv(data_dir / policy / campaign / f"{campaign}.csv")
    user_feat_cols = [c for c in df.columns if c.startswith("user_feature_")]
    context = pd.get_dummies(df[user_feat_cols], drop_first=True).to_numpy(dtype=np.float64)
    return {
        "n_rounds": len(df),
        "action": df["item_id"].to_numpy(dtype=np.int64),
        "reward": df["click"].to_numpy(dtype=np.float64),
        "pscore": df["propensity_score"].to_numpy(dtype=np.float64),
        "position": df["position"].to_numpy(dtype=np.int64),
        "context": context,
    }


random_logs = load_obd_logs("random", CAMPAIGN, DATA_DIR)
bts_logs = load_obd_logs("bts", CAMPAIGN, DATA_DIR)

items_df = pd.read_csv(DATA_DIR / "random" / CAMPAIGN / "item_context.csv")
N_ACTIONS = int(items_df["item_id"].nunique())
assert random_logs["action"].max() < N_ACTIONS and bts_logs["action"].max() < N_ACTIONS

print(f"campaign     = {CAMPAIGN}")
print(f"n_actions    = {N_ACTIONS}")
print(f"random logs  = {random_logs['n_rounds']} rounds")
print(f"bts logs     = {bts_logs['n_rounds']} rounds")
print(f"keys per log = {sorted(random_logs.keys())}")
print(f"context dim  = {random_logs['context'].shape[1]}")

campaign     = all
n_actions    = 80
random logs  = 10000 rounds
bts logs     = 10000 rounds
keys per log = ['action', 'context', 'n_rounds', 'position', 'pscore', 'reward']
context dim  = 20


## 4. Build the numpy-array bundle

scikit-rec's OPE evaluators are stateless and take **numpy arrays directly** — no `InteractionsDataset` wrapper for OPE. The shim is a six-array assembly:

| Array | Shape | OBD source |
| --- | --- | --- |
| `logged_items` | `(N, 1)` | `action` (already dense `0..n_actions-1`) |
| `logged_rewards` | `(N, 1)` | `reward` |
| `logging_proba` | `(N, 1)` | `pscore` |
| `recommendation_probas` | `(N, n_actions)` | target policy distribution per context — built in §5 |
| `recommendation_scores` | `(N, n_actions)` | identity copy of `recommendation_probas` (only `EXPECTED_REWARD` metric used; rank order doesn't matter) |
| `expected_rewards` | `(N, n_actions)` | reward-regressor predictions — built in §6 (DM/DR only) |

Each evaluator raises `ValueError` if a required array is missing; shape mismatches are caught by `_validate_input_shapes` in `BaseRecommenderEvaluator`.

In [4]:
def logs_to_arrays(logs: dict) -> dict:
    """Convert one obp BatchBanditFeedback dict to the (N, 1) arrays the
    scikit-rec OPE evaluators consume. Context is returned alongside for
    use by the §6 reward regressor and the §5 target-policy reconstruction.
    """
    n = logs["n_rounds"]
    actions = np.asarray(logs["action"], dtype=np.int64).reshape(n, 1)
    rewards = np.asarray(logs["reward"], dtype=np.float64).reshape(n, 1)
    pscore = np.asarray(logs["pscore"], dtype=np.float64).reshape(n, 1)
    context = np.asarray(logs["context"], dtype=np.float64)

    assert actions.min() >= 0 and actions.max() < N_ACTIONS, (
        f"action indices must be dense in [0, {N_ACTIONS - 1}]; got [{actions.min()}, {actions.max()}]"
    )
    assert (pscore > 0).all(), "logging propensities must be strictly positive"
    return {
        "logged_items": actions,
        "logged_rewards": rewards,
        "logging_proba": pscore,
        "context": context,
    }


random_arrays = logs_to_arrays(random_logs)
bts_arrays = logs_to_arrays(bts_logs)

for label, ar in [("random", random_arrays), ("bts", bts_arrays)]:
    print(
        f"{label:6s} | items {ar['logged_items'].shape} | rewards {ar['logged_rewards'].shape} "
        f"| pscore {ar['logging_proba'].shape} | context {ar['context'].shape}"
    )

random | items (10000, 1) | rewards (10000, 1) | pscore (10000, 1) | context (10000, 20)
bts    | items (10000, 1) | rewards (10000, 1) | pscore (10000, 1) | context (10000, 22)


## 5. Reconstruct target-policy distributions

Each evaluator (except ReplayMatch) needs `recommendation_probas: (N, n_actions)` — the target policy's full distribution over all actions for every logged context.

- **Target = Random:** uniform `1/n_actions` for every (context, item).
- **Target = BTS:** OBD's BTS is non-contextual (a stationary Beta–Binomial Thompson sampler over the action space, position-aware in expectation but stationary at the marginal level). For the §6.2 OPE evaluation we use the **empirical action distribution in BTS's own logs** as π_BTS — i.e. π_BTS(a) ≈ count(a in BTS logs) / N_bts — and broadcast this single distribution to every evaluation context. This is the marginal that `ExpectedRewardMetric` reports anyway, so the position dimension drops out cleanly.

In [5]:
def uniform_target_probas(n_rows: int) -> np.ndarray:
    """Random policy: uniform 1/n_actions over every action for every context."""
    return np.full((n_rows, N_ACTIONS), 1.0 / N_ACTIONS, dtype=np.float64)


def bts_target_probas(bts_logs: dict, n_rows_eval: int) -> np.ndarray:
    """BTS target policy reconstructed from the empirical action frequencies in BTS logs.

    OBD's BTS is a stationary Beta–Binomial Thompson sampler. Over the duration
    of the logging window, the marginal action distribution is what BTS's
    posterior converges to, which is what the OPE protocol's value estimate
    reports. Broadcast that single distribution to every evaluation context.
    """
    counts = np.bincount(bts_logs["action"], minlength=N_ACTIONS).astype(np.float64)
    p = counts / counts.sum()
    return np.tile(p, (n_rows_eval, 1))


# Primary direction: logging = random, target = bts.
# The symmetric direction (logging = bts, target = random) is built the same way.
n_eval = random_arrays["logged_items"].shape[0]
target_probas_random = uniform_target_probas(n_eval)
target_probas_bts = bts_target_probas(bts_logs, n_eval)

for label, p in [("random", target_probas_random), ("bts", target_probas_bts)]:
    assert p.shape == (n_eval, N_ACTIONS)
    assert np.allclose(p.sum(axis=1), 1.0), f"{label}: rows must sum to 1"
    print(
        f"target_probas_{label:6s}: shape {p.shape}, row-sum check OK, "
        f"entropy = {-(p[0] * np.log(np.clip(p[0], 1e-12, None))).sum():.3f} nats"
    )

target_probas_random: shape (10000, 80), row-sum check OK, entropy = 4.382 nats
target_probas_bts   : shape (10000, 80), row-sum check OK, entropy = 3.539 nats


## 6. Fit reward regressor for DM / DR

DR and DM both need `expected_rewards: (N, n_actions)` — a model's prediction of E[reward | context, action] for every (context × all actions) pair. We fit an XGBoost regressor on the logging-policy data using `(context_features, action_one_hot) → reward`, then predict over the cartesian product of evaluation contexts × all actions.

Skipped for IPS / SNIPS / ReplayMatch (they ignore `expected_rewards`).

In [6]:
def fit_reward_regressor(arrays: dict) -> xgb.XGBRegressor:
    """Fit a reward model on (context, action) -> reward over the logged policy."""
    context = arrays["context"]
    actions = arrays["logged_items"].ravel()
    rewards = arrays["logged_rewards"].ravel()

    action_oh = np.eye(N_ACTIONS, dtype=np.float64)[actions]
    X = np.hstack([context, action_oh])

    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=0,
        n_jobs=-1,
    )
    model.fit(X, rewards)
    return model


def predict_expected_rewards(model: xgb.XGBRegressor, eval_context: np.ndarray) -> np.ndarray:
    """Predict E[reward | context, item] for every (context, item) pair."""
    n = eval_context.shape[0]
    eye = np.eye(N_ACTIONS, dtype=np.float64)
    out = np.empty((n, N_ACTIONS), dtype=np.float64)
    for a in range(N_ACTIONS):
        action_oh = np.tile(eye[a], (n, 1))
        out[:, a] = model.predict(np.hstack([eval_context, action_oh]))
    return out


reward_model = fit_reward_regressor(random_arrays)
expected_rewards_random = predict_expected_rewards(reward_model, random_arrays["context"])
print(
    f"expected_rewards_random: shape {expected_rewards_random.shape}, "
    f"min {expected_rewards_random.min():.4g}, max {expected_rewards_random.max():.4g}"
)

expected_rewards_random: shape (10000, 80), min -0.04074, max 0.6362


## 7. Run all five evaluators

Each evaluator implements the same two-step contract:

1. `_compute_modified_rewards(...)` returns an `(N, n_actions)` array of modified rewards (NaN where unobserved or trimmed).
2. `evaluate(modified_rewards, ranks, scores, metric_type)` reduces to a scalar V̂.

We use `RecommenderMetricType.EXPECTED_REWARD` throughout — it's the metric the OBD protocol calls for, and it's what `SNIPSEvaluator`'s self-normalization assumes.

In [7]:
def run_all_evaluators(
    arrays: dict,
    target_probas: np.ndarray,
    expected_rewards: np.ndarray,
) -> dict[str, tuple[float, float]]:
    """Run all five OPE estimators on a single (logging, target) pair."""
    # ranks aren't used by EXPECTED_REWARD but the API still needs the array
    ranks = np.argsort(-target_probas, axis=1)
    scores = target_probas  # any monotonic transform works

    common_kwargs = dict(
        recommendation_scores=scores,
        recommendation_probas=target_probas,
        logged_items=arrays["logged_items"],
        logged_rewards=arrays["logged_rewards"],
    )
    estimators = {
        "IPS": (IPSEvaluator(), dict(logging_proba=arrays["logging_proba"])),
        "IPS-trim10": (IPSEvaluator(trim_threshold=10.0), dict(logging_proba=arrays["logging_proba"])),
        "SNIPS": (SNIPSEvaluator(), dict(logging_proba=arrays["logging_proba"])),
        "DR": (DREvaluator(), dict(logging_proba=arrays["logging_proba"], expected_rewards=expected_rewards)),
        "DM": (DirectMethodEvaluator(), dict(expected_rewards=expected_rewards)),
        "ReplayMatch": (ReplayMatchEvaluator(), {}),
    }
    out: dict[str, tuple[float, float]] = {}
    for name, (ev, extra) in estimators.items():
        modified = ev._compute_modified_rewards(**common_kwargs, **extra)
        # sanity check: silent zero on all-NaN is exactly the §7.2 invisible failure mode
        nan_frac = float(np.isnan(modified).mean())
        assert nan_frac < 1.0, f"{name}: every modified_reward is NaN (propensity wiring broken?)"
        v_hat = ev.evaluate(
            modified_rewards=modified,
            recommendation_ranks=ranks,
            recommendation_scores=scores,
            metric_type=RecommenderMetricType.EXPECTED_REWARD,
        )
        out[name] = (v_hat, nan_frac)
    return out


# Primary direction: logging policy = random, target = bts.
# Estimating V(π_bts) using random's logs, then comparing against V(π_bts) computed
# directly from bts's on-policy logs in §8.
results = run_all_evaluators(random_arrays, target_probas_bts, expected_rewards_random)
print(f"Estimating V(bts) from random logs (N = {random_arrays['logged_items'].shape[0]})\n")
for name, (v_hat, nan_frac) in results.items():
    print(f"  {name:13s}  V_hat = {v_hat:.6g}   nan_frac = {nan_frac:.3f}")

Estimating V(bts) from random logs (N = 10000)

  IPS            V_hat = 0.004668   nan_frac = 0.988
  IPS-trim10     V_hat = 0.004668   nan_frac = 0.988
  SNIPS          V_hat = 0.0048634   nan_frac = 0.988
  DR             V_hat = 0.00441839   nan_frac = 0.988
  DM             V_hat = 0.00421276   nan_frac = 0.000
  ReplayMatch    V_hat = 0.0038   nan_frac = 0.988


/Users/ssankararam/Shankar/Personal/RecSys/scikit-rec/skrec/evaluator/base_evaluator.py:160: UserWarning: Using Policy Metric 'EXPECTED_REWARD' with Evaluator 'ReplayMatchEvaluator'. Policy metrics typically expect a modified reward incorporating policy probability, not raw rewards.
  warnings.warn(


## 8. Compute ground-truth V(π_target) from on-policy logs

Per the OBD protocol: the ground-truth value of policy B is the empirical average reward in B's own logs. So when target = BTS, V(π_BTS) = mean of `bts_logs['reward']`. This is what every off-policy estimate is being compared against.

In [8]:
v_random_on_policy = float(np.mean(random_logs["reward"]))
v_bts_on_policy = float(np.mean(bts_logs["reward"]))
print(f"V(random) on-policy = {v_random_on_policy:.6g}")
print(f"V(bts)    on-policy = {v_bts_on_policy:.6g}")

V(random) on-policy = 0.0038
V(bts)    on-policy = 0.0042


## 9. Bias / variance / RMSE via bootstrap

Standard OBD reporting protocol: K=20 bootstrap subsamples of the logging-policy logs, recompute V̂ per estimator on each subsample, then report

- mean V̂ (point estimate),
- absolute bias |E[V̂] − V|,
- variance Var[V̂] across resamples,
- relative RMSE √(E[(V̂/V − 1)²]) — the OBD-paper headline metric.

If estimator orderings flip between bootstrap halves on the small-sample data, increase K or use the full 26M-row release.

In [9]:
def bootstrap_evaluators(
    arrays: dict,
    target_probas_full: np.ndarray,
    expected_rewards_full: np.ndarray,
    v_truth: float,
    k: int = 20,
    rng: np.random.Generator = RNG,
) -> pd.DataFrame:
    """Resample logged data K times; recompute every estimator on each resample.

    The reward regressor is NOT refit per bootstrap — that would dominate runtime
    and is not what the OBD paper does either. Variance reported here is over the
    OPE estimator + sampling, not over the regressor fit.
    """
    n = arrays["logged_items"].shape[0]
    by_name: dict[str, list[float]] = {}
    for _ in range(k):
        idx = rng.integers(0, n, size=n)
        sub = {key: arrays[key][idx] for key in ("logged_items", "logged_rewards", "logging_proba")}
        sub["context"] = arrays["context"][idx]
        sub_target = target_probas_full[idx]
        sub_exp = expected_rewards_full[idx]
        results = run_all_evaluators(sub, sub_target, sub_exp)
        for name, (v_hat, _) in results.items():
            by_name.setdefault(name, []).append(v_hat)

    rows = []
    for name, vals in by_name.items():
        vals = np.asarray(vals, dtype=np.float64)
        rel_err = vals / v_truth - 1.0
        rows.append(
            {
                "estimator": name,
                "V_hat_mean": vals.mean(),
                "abs_bias": abs(vals.mean() - v_truth),
                "variance": vals.var(ddof=1),
                "rel_rmse": np.sqrt(np.mean(rel_err**2)),
            }
        )
    return pd.DataFrame(rows).set_index("estimator")


# Random -> BTS: estimate V(π_bts) using random logs, compare against on-policy V(bts).
table = bootstrap_evaluators(
    random_arrays,
    target_probas_bts,
    expected_rewards_random,
    v_truth=v_bts_on_policy,
    k=20,
)
print(f"Random -> BTS  |  v_truth = V(bts) on-policy = {v_bts_on_policy:.6g}\n")
table

Random -> BTS  |  v_truth = V(bts) on-policy = 0.0042



,V_hat_mean,abs_bias,variance,rel_rmse
estimator,,,,
IPS,0.004174,0.000026,9.878895e-07,0.230741
IPS-trim10,0.004174,0.000026,9.878895e-07,0.230741
SNIPS,0.004349,0.000149,1.145578e-06,0.250902
DR,0.003932,0.000268,8.291318e-07,0.220720
DM,0.004206,0.000006,1.030674e-09,0.007581
ReplayMatch,0.003560,0.000640,2.846316e-07,0.196338


## 10. Cross-check against `obp` reference estimators

scikit-rec and obp implement the same OPE math, so on identical inputs their estimates should agree to within numerical noise (~1%). Disagreement larger than that is a scikit-rec bug worth filing.

obp's API is position-aware: `action_dist` has shape `(n, n_actions, len_list)`. We collapse to `len_list = 1` and pass `position=None` since the §6.2 evaluation reports a marginal value, not per-position values. Note that obp's `DirectMethod` uses a slightly different normalization than scikit-rec (no `n_actions` rescale), so the absolute numbers differ — what matters is that the DR / IPS / SNIPS columns line up.

In [10]:
from obp.ope import (
    DirectMethod as ObpDirectMethod,
)
from obp.ope import (
    DoublyRobust as ObpDoublyRobust,
)
from obp.ope import (
    InverseProbabilityWeighting as ObpIPS,
)
from obp.ope import (
    SelfNormalizedInverseProbabilityWeighting as ObpSNIPS,
)

action_dist_obp = target_probas_bts.reshape(-1, N_ACTIONS, 1)
estimated_rewards_obp = expected_rewards_random.reshape(-1, N_ACTIONS, 1)

obp_estimates = {
    "IPS": ObpIPS().estimate_policy_value(
        reward=random_logs["reward"],
        action=random_logs["action"],
        pscore=random_logs["pscore"],
        action_dist=action_dist_obp,
    ),
    "SNIPS": ObpSNIPS().estimate_policy_value(
        reward=random_logs["reward"],
        action=random_logs["action"],
        pscore=random_logs["pscore"],
        action_dist=action_dist_obp,
    ),
    "DR": ObpDoublyRobust().estimate_policy_value(
        reward=random_logs["reward"],
        action=random_logs["action"],
        pscore=random_logs["pscore"],
        action_dist=action_dist_obp,
        estimated_rewards_by_reg_model=estimated_rewards_obp,
    ),
    "DM": ObpDirectMethod().estimate_policy_value(
        action_dist=action_dist_obp,
        estimated_rewards_by_reg_model=estimated_rewards_obp,
    ),
}

# Single-shot scikit-rec results on the full random logs (no bootstrap).
skrec_estimates = {name: v for name, (v, _) in results.items() if name in obp_estimates}

cmp = pd.DataFrame(
    {
        "skrec": pd.Series(skrec_estimates),
        "obp": pd.Series(obp_estimates),
    }
)
cmp["abs_diff"] = (cmp["skrec"] - cmp["obp"]).abs()
cmp["rel_diff"] = cmp["abs_diff"] / cmp["obp"].abs().clip(lower=1e-12)
cmp

,skrec,obp,abs_diff,rel_diff
IPS,0.004668,0.004668,0.000000e+00,0.000000e+00
SNIPS,0.004863,0.004863,1.734723e-18,3.566898e-16
DR,0.004418,0.004418,0.000000e+00,0.000000e+00
DM,0.004213,0.004213,0.000000e+00,0.000000e+00


## 11. Follow-ups (out of scope for this scaffold)

1. **Symmetric direction (BTS → Random).** Mirror §4–§9 with `bts_arrays`, `target_probas_random`, `v_random_on_policy`. The reward regressor must be refit on `bts_arrays` since it should always train on the logging policy.
2. **Three-campaign sweep.** Loop over `CAMPAIGN ∈ {'men', 'women', 'all'}` and concatenate the bias / variance / RMSE tables to fill the six-cell experiment grid the plan calls for.
3. **Full 26M-row release.** The notebook works on the bundled ~10K/campaign sample; running on the full release at https://research.zozo.com/data.html would tighten the bootstrap intervals.